# Hall-Petch Relation — Linear Regression Practical
### Day 1 Case Study | FDP: Machine Learning for Materials and Metallurgical Engineering

**Background:** The Hall-Petch relation describes how a metal's yield strength increases as its grain size decreases:

$$\sigma_y = \sigma_0 + k\,d^{-1/2}$$

where:
- $\sigma_y$ = yield stress (MPa)
- $d$ = average grain diameter (mm)
- $\sigma_0$ = "friction stress" — the yield stress of a single crystal (theoretical grain size → ∞)
- $k$ = strengthening coefficient — how strongly grain boundaries block dislocation motion

This is **nonlinear** in $d$, but becomes **linear** once we plot $\sigma_y$ against $d^{-1/2}$ instead of $d$ itself — a classic linearization. That lets us use ordinary linear regression to find $\sigma_0$ (intercept) and $k$ (slope) directly from experimental data.

## Step 1 — Load the data
Grain size and yield stress measurements (7 data points), loaded directly from the course GitHub repo -- no file upload needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Loaded directly from the course repo -- update this URL if you rename the repo
DATA_URL = "https://raw.githubusercontent.com/vijindal/fdp-ml/main/notebooks/day1_hallpetch.csv"
df = pd.read_csv(DATA_URL)
df


## Step 2 — Linearize
Compute $d^{-1/2}$ so the relationship becomes linear in this transformed variable.

In [ ]:
df["d_inv_sqrt"] = df["grain_size_mm"] ** -0.5
df


## Step 3 — Visualize
First, look at the raw (nonlinear) relationship, then the linearized version.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(df["grain_size_mm"], df["yield_stress_MPa"], color="darkred")
axes[0].set_xlabel("Grain size, d (mm)")
axes[0].set_ylabel(r"Yield stress, $\sigma_y$ (MPa)")
axes[0].set_title("Raw relationship (nonlinear)")

axes[1].scatter(df["d_inv_sqrt"], df["yield_stress_MPa"], color="navy")
axes[1].set_xlabel("$d^{-1/2}$ (mm$^{-1/2}$)")
axes[1].set_ylabel(r"Yield stress, $\sigma_y$ (MPa)")
axes[1].set_title("Linearized relationship")

plt.tight_layout()
plt.show()


## Step 4 — Fit a straight line (least-squares)
Use `np.polyfit` to fit $\sigma_y = \sigma_0 + k\,(d^{-1/2})$.

In [ ]:
# np.polyfit(x, y, degree=1) returns [slope, intercept]
k, sigma0 = np.polyfit(df["d_inv_sqrt"], df["yield_stress_MPa"], 1)

print(f"Fitted Hall-Petch equation:")
print(f"  sigma_y = {sigma0:.2f} + {k:.2f} * d^(-1/2)")
print()
print(f"sigma_0 (friction stress)      = {sigma0:.2f} MPa")
print(f"k (strengthening coefficient)  = {k:.2f} MPa*mm^0.5")


In [ ]:
# Plot the fit over the linearized data
x_line = np.linspace(df["d_inv_sqrt"].min(), df["d_inv_sqrt"].max(), 100)
y_line = sigma0 + k * x_line

plt.figure(figsize=(6, 4.5))
plt.scatter(df["d_inv_sqrt"], df["yield_stress_MPa"], color="navy", label="Experimental data")
plt.plot(x_line, y_line, color="crimson", label="Least-squares fit")
plt.xlabel("$d^{-1/2}$ (mm$^{-1/2}$)")
plt.ylabel(r"Yield stress, $\sigma_y$ (MPa)")
plt.title("Hall-Petch fit")
plt.legend()
plt.tight_layout()
plt.show()


## Step 5 — Interpret the fit physically

- **$\sigma_0$** tells us the intrinsic lattice resistance to dislocation motion — the strength we'd expect even with infinitely large grains (no grain-boundary strengthening at all).
- **$k$** tells us how effectively grain boundaries block dislocations — a larger $k$ means grain refinement is a more powerful strengthening lever for this material.

Compare your fitted values against literature values for this material family — are they in a physically reasonable range?

## Step 6 — Use the model: predict yield stress for a new grain size
Estimate $\sigma_y$ for a specimen with grain size $d = 0.003$ mm (not in the original dataset).

In [ ]:
d_new = 0.003
sigma_y_pred = sigma0 + k * (d_new ** -0.5)
print(f"Predicted yield stress at d = {d_new} mm: {sigma_y_pred:.1f} MPa")


---
## (Optional) Stretch — fit by gradient descent instead of `np.polyfit`

`np.polyfit` solves the least-squares problem directly. If you'd like to connect this back to the gradient-descent derivation from the morning's ML-landscape lecture, here's the same fit done iteratively instead.

In [ ]:
x = df["d_inv_sqrt"].values
y = df["yield_stress_MPa"].values
n = len(x)

w, b = 0.0, 0.0          # initial weight (k) and bias (sigma0)
lr = 0.001               # learning rate (kept small -- these features aren't scaled, see note below)
epochs = 100000

for _ in range(epochs):
    y_pred = w * x + b
    dw = (-2/n) * np.sum(x * (y - y_pred))
    db = (-2/n) * np.sum(y - y_pred)
    w -= lr * dw
    b -= lr * db

print(f"Gradient descent result:  k = {w:.2f}, sigma_0 = {b:.2f}")
print(f"np.polyfit result:        k = {k:.2f}, sigma_0 = {sigma0:.2f}")


**Note on the learning rate:** $d^{-1/2}$ ranges from 2 to about 23.6 here, unscaled. A learning rate as small as 0.001 is needed for convergence — try lr = 0.01 yourself and watch it diverge (this is a good live demonstration of the "Learning Rate" and "Feature Scaling" slides from the morning lecture: too-high a learning rate on unscaled features causes the loss to blow up rather than converge).